# Getting started with the Celeris API

Celeris serves **celeris-1**, a low-latency diffusion LLM, through an
OpenAI-compatible API — so the official OpenAI SDK *is* the Celeris client.
This notebook makes your first chat completion and covers the two things every
Celeris client must get right:

1. **`max_tokens` normalization.** The wire `max_tokens` value must be a
   multiple of 256 in the range 256–1024. Anything else is rejected, so round
   your requested output budget up to the nearest supported step.
2. **Input budgeting.** Requests share a **4096-token total window** between
   input and output. Reserving `max_tokens` for output leaves
   `4096 - max_tokens` tokens for all input messages combined.

## Setup

```sh
pip install openai
export CELERIS_BASE_URL="https://<your-celeris-endpoint>"
export CELERIS_API_KEY="<your-api-key>"
```

In [ ]:
import os

from openai import OpenAI

client = OpenAI(
    # The OpenAI SDK reads OPENAI_BASE_URL / OPENAI_API_KEY by default;
    # prefer the CELERIS_* names, falling back to the SDK defaults.
    base_url=os.environ.get("CELERIS_BASE_URL") or os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("CELERIS_API_KEY") or os.environ.get("OPENAI_API_KEY"),
)
MODEL = os.environ.get("CELERIS_MODEL", "celeris-1")

## The token window

Celeris requests live in a 4096-token total window. `max_tokens` reserves the
output canvas — and it only accepts multiples of 256 between 256 and 1024.
These two helpers are the canonical budgeting code used throughout this
cookbook; copy them into your own client.

In [ ]:
TOTAL_TOKEN_LIMIT = 4096
MIN_OUTPUT_TOKENS = 256
MAX_OUTPUT_TOKENS = 1024
OUTPUT_TOKEN_STEP = 256


def normalize_max_tokens(requested: int) -> int:
    """Round up to a supported 256-token step, clamped to [256, 1024].

    Rounding up avoids silently giving yourself less output room than you
    asked for.
    """
    if requested <= MIN_OUTPUT_TOKENS:
        return MIN_OUTPUT_TOKENS
    if requested >= MAX_OUTPUT_TOKENS:
        return MAX_OUTPUT_TOKENS
    step = OUTPUT_TOKEN_STEP
    return ((requested + step - 1) // step) * step


def input_token_budget(max_tokens: int) -> int:
    """Tokens left for ALL input messages after the output reservation."""
    return TOTAL_TOKEN_LIMIT - normalize_max_tokens(max_tokens)


for requested in (100, 300, 512, 2000):
    normalized = normalize_max_tokens(requested)
    print(f"requested {requested:>4} -> max_tokens {normalized:>4}, "
          f"input budget {input_token_budget(requested)} tokens")

## Fitting the prompt

Celeris does not expose its tokenizer, so estimate ~4 bytes per token plus a
small allowance for the chat template, and clip long input *before* sending —
the API will not do it for you.

In [ ]:
ESTIMATED_BYTES_PER_TOKEN = 4
CHAT_BASE_TOKENS = 8
CHAT_MESSAGE_TOKENS = 4


def fit_input(system: str, user: str, max_tokens: int) -> str:
    """Clip the user message so the full request fits the 4096-token window."""
    overhead = CHAT_BASE_TOKENS + 2 * CHAT_MESSAGE_TOKENS
    system_tokens = -(-len(system.encode("utf-8")) // ESTIMATED_BYTES_PER_TOKEN)
    user_budget_tokens = input_token_budget(max_tokens) - overhead - system_tokens
    max_user_bytes = max(user_budget_tokens, 0) * ESTIMATED_BYTES_PER_TOKEN
    encoded = user.encode("utf-8")
    if len(encoded) <= max_user_bytes:
        return user
    return encoded[:max_user_bytes].decode("utf-8", errors="ignore")

## Your first completion

Everything else is standard OpenAI SDK usage.

In [ ]:
system = "You are a concise assistant. Answer directly."
user = "Explain what a context window is, in two sentences."

max_tokens = normalize_max_tokens(512)
user = fit_input(system, user, max_tokens)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ],
    max_tokens=max_tokens,
    temperature=0.2,
)

print(response.choices[0].message.content.strip())
if response.usage:
    print(f"\n[usage: {response.usage.prompt_tokens} prompt + "
          f"{response.usage.completion_tokens} completion tokens]")

## Streaming

Celeris's headline feature is speed. Stream to see it: time-to-first-token is
the number to watch.

In [ ]:
import time

started = time.monotonic()
first_token_at = None

stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "List three uses for a fast, small LLM."}],
    max_tokens=normalize_max_tokens(256),
    temperature=0.2,
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None
    if delta:
        if first_token_at is None:
            first_token_at = time.monotonic()
        print(delta, end="", flush=True)

print()
if first_token_at is not None:
    print(f"\n[first token in {(first_token_at - started) * 1000:.0f} ms, "
          f"total {(time.monotonic() - started) * 1000:.0f} ms]")

## Next steps

- Tune `max_tokens` to your use case: summaries usually fit in 256; generation
  tasks may want the full 1024 — at the cost of 768 fewer input tokens.
- See the [chatbot example](../chatbot/) for a streaming web app that
  showcases latency, and the
  [alert triage agent](../alert-triage-agent/) for a multi-step tool-use loop
  under the same token budget.